In [67]:
# Imports base
import os
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [68]:
# Leitura e transformação
import geopandas as gpd
from shapely import wkt

gdf_alagamento = gpd.read_file('risco_ocorrencias/risco_ocorrencia_alagamento_v2.csv')
gdf_inundacao  = gpd.read_file('risco_ocorrencias/risco_ocorrencia_inundacao_v2.csv')

gdf_ocorrencias = pd.concat(
    [gdf_alagamento, gdf_inundacao],
    ignore_index=True
)

gdf_ocorrencias = gdf_ocorrencias.drop(columns='FID')

gdf_ocorrencias['geometry'] = gdf_ocorrencias['ge_ponto'].apply(wkt.loads)
gdf_ocorrencias = gpd.GeoDataFrame(gdf_ocorrencias, geometry='geometry', crs='EPSG:31983')

gdf_ocorrencias = gdf_ocorrencias.to_crs(epsg=4326)
gdf_zonas = gpd.read_file('../zonas_sp.geojson').to_crs(epsg=4326)
gdf_zonas = gdf_zonas[['zona', 'geometry']].copy()

gdf_ocorrencias = gdf_ocorrencias.sjoin(
    gdf_zonas,
    how='left',
    predicate='within'
).drop(columns='index_right', errors='ignore')

gdf_ocorrencias['longitude'] = gdf_ocorrencias.geometry.x
gdf_ocorrencias['latitude']  = gdf_ocorrencias.geometry.y

gdf_ocorrencias['dt_ocorrencia'] = pd.to_datetime(gdf_ocorrencias['dt_ocorrencia'])
gdf_ocorrencias = gdf_ocorrencias.sort_values('dt_ocorrencia', ascending=True).reset_index(drop=True)

gdf_ocorrencias['latitude']  = gdf_ocorrencias['latitude'].round(6)
gdf_ocorrencias['longitude'] = gdf_ocorrencias['longitude'].round(6)

print("Colunas:", gdf_ocorrencias.columns.tolist())
print(gdf_ocorrencias[['cd_identificador', 'latitude', 'longitude', 'zona']].head())
print(gdf_ocorrencias['zona'].value_counts(dropna=False))
print(f"NaN em zona: {gdf_ocorrencias['zona'].isna().sum()}")
print(f"Total: {len(gdf_ocorrencias)}")

Colunas: ['cd_identificador', 'dt_ocorrencia', 'dc_tipo_ocorrencia', 'nm_distrito', 'ge_ponto', 'dt_carga', 'nm_subprefeitura', 'sg_fonte_original', 'geometry', 'zona', 'longitude', 'latitude']
  cd_identificador   latitude  longitude   zona
0               19 -23.819657 -46.702602    Sul
1               30 -23.819500 -46.702627    Sul
2               28 -23.819656 -46.702602    Sul
3               53 -23.715772 -46.652718    Sul
4               71 -23.591357 -46.383584  Leste
zona
Leste     266
Sul       242
Norte      45
Oeste      30
Centro      3
Name: count, dtype: int64
NaN em zona: 0
Total: 586


In [69]:
# Export CSV
df = gdf_ocorrencias.drop(columns=['nm_distrito', 'geometry'], errors='ignore')

df.to_csv(
    'saidas/ocorrencias.csv',
    index=False,
    encoding='utf-8-sig',
    sep=';',
    decimal=','
)

In [ ]:
# Setup dos engines
from sqlalchemy import create_engine

for l in Path(".env").read_text(encoding="utf-8").splitlines():
    if "=" in l and not l.startswith("#"):
        k, v = l.split("=", 1)
        os.environ[k.strip()] = v.strip().strip('"').strip("'")

engine_local = create_engine(os.environ["LOCAL_DB_URL"])
engine_cloud = create_engine(os.environ["SUPABASE_DB_URL"])

In [ ]:
# Persistência nos bancos
df = pd.read_csv('saidas/ocorrencias.csv', sep=';', encoding='utf-8-sig', decimal=',')

df.to_sql('df_ocorrencias', engine_local, if_exists='replace', index=False )
df.to_sql('df_ocorrencias', engine_cloud, if_exists='replace', index=False)
print('ok')

In [70]:
# Diagnóstico dos CSVs do CEMADEN
import glob

for arquivo in sorted(glob.glob('Cemaden/data_*.csv')):
    with open(arquivo, 'r', encoding='latin-1') as f:
        primeira = f.readline()
    print(f"{arquivo}: {primeira[:200]!r}")

Cemaden\data_ABR.csv: 'municipio\tcodEstacao\tuf\tnomeEstacao\tlatitude\tlongitude\tdatahora\tvalorMedida\n'
Cemaden\data_AGO.csv: 'ï»¿municipio;codEstacao;uf;nomeEstacao;latitude;longitude;datahora;valorMedida\n'
Cemaden\data_FEV.csv: 'ï»¿municipio;codEstacao;uf;nomeEstacao;latitude;longitude;datahora;valorMedida\n'
Cemaden\data_JAN.csv: 'ï»¿municipio;codEstacao;uf;nomeEstacao;latitude;longitude;datahora;valorMedida\n'
Cemaden\data_JLH.csv: 'ï»¿municipio;codEstacao;uf;nomeEstacao;latitude;longitude;datahora;valorMedida\n'
Cemaden\data_JUN.csv: 'ï»¿municipio;codEstacao;uf;nomeEstacao;latitude;longitude;datahora;valorMedida\n'
Cemaden\data_MAI.csv: 'ï»¿municipio;codEstacao;uf;nomeEstacao;latitude;longitude;datahora;valorMedida\n'
Cemaden\data_MAR.csv: 'ï»¿municipio;codEstacao;uf;nomeEstacao;latitude;longitude;datahora;valorMedida\n'
Cemaden\data_OUT.csv: 'ï»¿municipio;codEstacao;uf;nomeEstacao;latitude;longitude;datahora;valorMedida\n'
Cemaden\data_SET.csv: 'ï»¿municipio;codEstacao;uf;n

In [71]:
arquivos_cemaden = sorted(glob.glob('Cemaden/data_*.csv'))
lista_dfs = []

for arquivo in arquivos_cemaden:
    # 1) Lê a primeira linha tentando utf-8-sig, com fallback para latin-1
    try:
        with open(arquivo, 'r', encoding='utf-8-sig') as f:
            primeira = f.readline()
        encoding = 'utf-8-sig'
    except UnicodeDecodeError:
        with open(arquivo, 'r', encoding='latin-1') as f:
            primeira = f.readline()
        encoding = 'latin-1'

    # 2) Detecta separador
    sep = '\t' if '\t' in primeira else ';'

    # 3) Lê com o encoding e separador detectados
    df = pd.read_csv(arquivo, sep=sep, encoding=encoding, decimal=',')

    df['arquivo_origem'] = arquivo
    lista_dfs.append(df)
    print(f"OK  {arquivo}: enc={encoding}, sep={sep!r}, {len(df)} linhas, {len(df.columns)} colunas")

df_cemaden = pd.concat(lista_dfs, ignore_index=True)
print(f"\nTotal: {len(df_cemaden)} registros")
print(f"Colunas: {df_cemaden.columns.tolist()}")

OK  Cemaden\data_ABR.csv: enc=latin-1, sep='\t', 45720 linhas, 9 colunas
OK  Cemaden\data_AGO.csv: enc=utf-8-sig, sep=';', 44867 linhas, 9 colunas
OK  Cemaden\data_FEV.csv: enc=utf-8-sig, sep=';', 53655 linhas, 9 colunas
OK  Cemaden\data_JAN.csv: enc=utf-8-sig, sep=';', 61193 linhas, 9 colunas
OK  Cemaden\data_JLH.csv: enc=utf-8-sig, sep=';', 43320 linhas, 9 colunas
OK  Cemaden\data_JUN.csv: enc=utf-8-sig, sep=';', 51982 linhas, 9 colunas
OK  Cemaden\data_MAI.csv: enc=utf-8-sig, sep=';', 50405 linhas, 9 colunas
OK  Cemaden\data_MAR.csv: enc=utf-8-sig, sep=';', 54630 linhas, 9 colunas
OK  Cemaden\data_OUT.csv: enc=utf-8-sig, sep=';', 53569 linhas, 9 colunas
OK  Cemaden\data_SET.csv: enc=utf-8-sig, sep=';', 65026 linhas, 9 colunas

Total: 524367 registros
Colunas: ['municipio', 'codEstacao', 'uf', 'nomeEstacao', 'latitude', 'longitude', 'datahora', 'valorMedida', 'arquivo_origem']


In [72]:
# Contagem por arquivo (todos devem ter as 8 colunas originais)
print(df_cemaden.groupby('arquivo_origem').agg(
    linhas=('municipio','size'),
    colunas=('municipio', lambda s: df_cemaden.shape[1])
))

# 2) Checagem de nulos nas colunas-chave
print(df_cemaden[['municipio','uf','datahora','valorMedida']].isna().sum())

# 3) Amostra do ABR (para ver se os acentos ficaram corretos com latin-1)
print(df_cemaden[df_cemaden['arquivo_origem'].str.contains('ABR')]['municipio'].head(10))

                      linhas  colunas
arquivo_origem                       
Cemaden\data_ABR.csv   45720        9
Cemaden\data_AGO.csv   44867        9
Cemaden\data_FEV.csv   53655        9
Cemaden\data_JAN.csv   61193        9
Cemaden\data_JLH.csv   43320        9
Cemaden\data_JUN.csv   51982        9
Cemaden\data_MAI.csv   50405        9
Cemaden\data_MAR.csv   54630        9
Cemaden\data_OUT.csv   53569        9
Cemaden\data_SET.csv   65026        9
municipio      0
uf             0
datahora       0
valorMedida    0
dtype: int64
0    SÃO PAULO
1    SÃO PAULO
2    SÃO PAULO
3    SÃO PAULO
4    SÃO PAULO
5    SÃO PAULO
6    SÃO PAULO
7    SÃO PAULO
8    SÃO PAULO
9    SÃO PAULO
Name: municipio, dtype: str


In [74]:
# Zonas — carrega uma vez
import geopandas as gpd

gdf_zonas = gpd.read_file('../zonas_sp.geojson').to_crs(epsg=4326)

coluna_zona = next(
    (c for c in gdf_zonas.columns
     if c.lower() in ['nome_zona','zona','nm_zona','nome_distrito','distrito','regiao','nome_regiao']),
    [c for c in gdf_zonas.columns if c != 'geometry'][0],
)
print(f"Zonas: {len(gdf_zonas)} | coluna usada: '{coluna_zona}'")

Zonas: 5 | coluna usada: 'zona'
